IMPORTS

In [ ]:
import os
import os.path as op

import openneuro
from mne.datasets import sample

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)

In [ ]:
import os
from os import path
import openneuro
from mne.datasets import sample
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.viz import plot_topomap

from naplib.io import load_bids

In [ ]:
import mne  # Import MNE

# Download the sample dataset if it is not already downloaded
data_path = mne.datasets.sample.data_path(download=True)

print("MNE sample dataset path:", data_path)

Download EEG data from OpenNeuro

Import data and get auditory spectrogram which will be used as stimulus.


In [ ]:
dataset = 'ds002778'
subject = 'pd12'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

Read the data into a Data object

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

Compute Alpha Theta Ratio

Let's compute the Alpha/Theta Ratio in each channel. We use the log-value so that ratios above 1 are positive and ratios below 1 are negative, which makes the resulting topomap more clear.

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

Visualize Results with MNE

The Data contains the mne_info attribute (data.mne_info) which we can use for plotting
This info is an instance of mne.Info, and it contains measurement information
like channel names, locations, etc, as well as other metadata

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
!ls /Documents/PD_EEG_TUTORIAL/read_PD_datasets/ds002778-download/sub-hc24/ses-hc/eeg/sub-hc1_ses-hc_task-rest_eeg.bdf/

In [ ]:
print(os.listdir("/Users/uarc/Downloads/ds002778/sub-hc1/ses-hc/eeg/sub-hc1_ses-hc_task-rest_eeg.bdf"))

In [ ]:
import pandas as pd
import mne
import numpy as np

# Define the path to your events file
events_file = "/Users/uarc/Downloads/ds002778"

# Load the events file using pandas
events_df = pd.read_csv(events_file, sep="\t")

# Check the structure of the file
print(events_df.head())

# Convert the event data into MNE format
# MNE expects events as a NumPy array with three columns: (sample, 0, event_id)
events = np.column_stack((events_df["onset"], np.zeros(len(events_df), dtype=int), events_df["value"]))

# Ensure the event timestamps are in integer samples
events[:, 0] = events[:, 0].astype(int)

# Print the first few events
print(events[:5])


In [ ]:
ls /Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-pd12/


In [ ]:
import mne

# Define the correct file path
bdf_file = "/Users/uarc/Downloads/ds002778f"

# Load BDF file
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Print basic information
print(raw.info)


In [ ]:
import pandas as pd

# Load events file
events_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-pd12/ses-off/eeg/sub-pd12_ses-off_task-rest_events.tsv"
events = pd.read_csv(events_file, sep="\t")

print(events.head())  # Check the first few rows


In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-pd12/ses-off/eeg/sub-pd12_ses-off_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'pd22'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-pd22/ses-off/eeg/sub-pd22_ses-off_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'pd9'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-pd22/ses-off/eeg/sub-pd22_ses-off_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


Download Healthy Participant Data

In [ ]:
dataset = 'ds002778'
subject = 'hc24'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc24",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
ls Documents/PD_EEG_TUTORIAL/read_PD_datasets/ds002778-download/sub-hc24/ses-hc/eeg

In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/read_PD_datasets/ds002778-download/sub-hc24/ses-hc/eeg/sub-hc24_ses-hc_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)

# Optional: Check the filtered data
raw.plot(duration=30, n_channels=30, scalings='auto')


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'hc29'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc29",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-hc29/ses-hc/eeg/sub-hc29_ses-hc_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)

# Optional: Check the filtered data
raw.plot(duration=30, n_channels=30, scalings='auto')


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()

In [ ]:
dataset = 'ds002778'
subject = 'hc33'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc33",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define file path
bdf_file = "/Users/calebcooper/Documents/PD_EEG_TUTORIAL/ds002778/sub-hc29/ses-hc/eeg/sub-hc29_ses-hc_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)

# Optional: Check the filtered data
raw.plot(duration=30, n_channels=30, scalings='auto')

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()
